# Data Conversion

This notebook contains the code for the data conversion described in Section 4 of the project documentation including reading the brat-formatted files and converting them to txt-files containing triples of IDs inspired by CoDEx.

In [3]:
# install requirements if necessary

#!pip install mendelai-brat-parser==0.0.11 scikit-learn==1.7.2

In [4]:
# import modules

import os
import json

from sklearn.model_selection import train_test_split
from brat_parser import get_entities_relations_attributes_groups

In [6]:
# set paths to files and folders

corpusdir = "brat_formatted_PICKLE_dataset_unsplit"
testfile = "brat_formatted_PICKLE_dataset_unsplit/PMID84635_abstract.ann"

outfile_entities = "codex_formatted_PICKLE_unsplit/entities.json"
outfile_relations = "codex_formatted_PICKLE_unsplit/relations.json"
outfile_train = "codex_formatted_PICKLE_split/train.txt"
outfile_val = "codex_formatted_PICKLE_split/valid.txt"
outfile_test = "codex_formatted_PICKLE_split/test.txt"

In [7]:
# explore PICKLE format

e, r, a, g = get_entities_relations_attributes_groups(testfile)

print(f"entities = {e}")
print(f"relations = {r}")
print(f"attributes = {a}")
print(f"groups = {g}")

entities = {'T1': Entity(id='T1', type='Plant_hormone', span=((0, 16),), text='Gibberellic acid'), 'T2': Entity(id='T2', type='Plant_hormone', span=((18, 23),), text='GA(3)'), 'T3': Entity(id='T3', type='Biochemical_process', span=((77, 91),), text='salt tolerance'), 'T4': Entity(id='T4', type='Inorganic_compound_other', span=((109, 113),), text='salt'), 'T5': Entity(id='T5', type='Plant_hormone', span=((198, 203),), text='GA(3)'), 'T6': Entity(id='T6', type='Inorganic_compound_other', span=((218, 222),), text='salt'), 'T7': Entity(id='T7', type='Multicellular_organism', span=((290, 303),), text='oat cultivars'), 'T8': Entity(id='T8', type='Multicellular_organism', span=((305, 309),), text='Oats'), 'T9': Entity(id='T9', type='Inorganic_compound_other', span=((344, 348),), text='salt'), 'T10': Entity(id='T10', type='Multicellular_organism', span=((354, 359),), text='wheat'), 'T11': Entity(id='T11', type='Multicellular_organism', span=((364, 370),), text='barley'), 'T12': Entity(id='T12'

In [8]:
# convert entities to CoDEx-like format

def get_entities(entities, id = 0, ent_set = set()):
    entities_codex = {}
    mapping = {}

    for ent in entities.values():
        ent_set.add((ent.id, ent.text, ent.type))

    for e in ent_set:
        id += 1
        entities_codex[f"E{id}"] = {"label": e[1], "description": e[2]}
        mapping[e[0]] = f"E{id}"

    return entities_codex, mapping


In [9]:
# convert relations to CoDEx-like format

def get_relations(relations, id = 0, rel_set = set()):
    relations_codex = {}

    for rel in relations.values():
        rel_set.add(rel.type)

    for r in rel_set:
        id += 1
        relations_codex[f"R{id}"] = {"label": r}

    rel_to_id = {rel["label"] : id for id, rel in relations_codex.items()}

    return relations_codex, rel_to_id


In [10]:
# convert triples to CoDEx-like format

def get_triples(relations, mapping, rel_to_id):
    triples_codex = []

    for rel in relations.values():
        triples_codex.append(f"{mapping[rel.subj]}\t{rel_to_id[rel.type]}\t{mapping[rel.obj]}\n")

    return triples_codex


In [11]:
# visualise entities, relations, and triples in CoDEx-like format for testfile

entities_codex, mapping = get_entities(e)
print(f"entities_codex = {entities_codex}")
print(f"mapping = {mapping}")

relations_codex, rel_to_id = get_relations(r)
print(f"relations_codex = {relations_codex}")
print(f"mapping = {mapping}")

triples_codex = get_triples(r, mapping, rel_to_id)
print(f"triples_codex = {triples_codex}")

entities_codex = {'E1': {'label': 'sensitive oat cultivars', 'description': 'Multicellular_organism'}, 'E2': {'label': 'GA(3)', 'description': 'Plant_hormone'}, 'E3': {'label': 'GA(3)', 'description': 'Plant_hormone'}, 'E4': {'label': 'GA(3)', 'description': 'Plant_hormone'}, 'E5': {'label': 'UPO-94', 'description': 'Multicellular_organism'}, 'E6': {'label': 'salt', 'description': 'Inorganic_compound_other'}, 'E7': {'label': 'NDO-2', 'description': 'Multicellular_organism'}, 'E8': {'label': 'salt tolerance', 'description': 'Biochemical_process'}, 'E9': {'label': 'Oats', 'description': 'Multicellular_organism'}, 'E10': {'label': 'salt', 'description': 'Inorganic_compound_other'}, 'E11': {'label': 'oat cultivars', 'description': 'Multicellular_organism'}, 'E12': {'label': 'wheat', 'description': 'Multicellular_organism'}, 'E13': {'label': 'salt', 'description': 'Inorganic_compound_other'}, 'E14': {'label': 'UPO-212', 'description': 'Multicellular_organism'}, 'E15': {'label': 'Gibberellic

In [12]:
## get entities, relations, and triples from all .ann files

# initialise
entities, relations, triples = {}, {}, []
ent_id, rel_id = 0, 0
ent_set, rel_set = set(), set()

# iterate over all files in corpus directory
for file in os.listdir(corpusdir):
    
    # check if file contains annotations
    if file.endswith(".ann"):

        print(f"parsing {file}...")
        filepath = os.path.join(corpusdir, file)

        # parse BRAT format
        ent, rel, _, _ = get_entities_relations_attributes_groups(filepath)

        # get entities, relations, triples
        entities_file, mapping = get_entities(ent, ent_id, ent_set)
        relations_file, rel_to_id = get_relations(rel, rel_id, rel_set)
        triples_file = get_triples(rel, mapping, rel_to_id)

        # add entities, relations, triples of current file to collection
        entities |= entities_file
        relations |= relations_file
        triples.extend(triples_file)


parsing PMID10072406_abstract.ann...
parsing PMID10198087_abstract.ann...
parsing PMID10476063_abstract.ann...
parsing PMID10549554_abstract.ann...
parsing PMID10580327_abstract.ann...
parsing PMID10707361_abstract.ann...
parsing PMID10890883_abstract.ann...
parsing PMID10972869_abstract.ann...
parsing PMID11094982_abstract.ann...
parsing PMID11278736_abstract.ann...
parsing PMID11299384_abstract.ann...
parsing PMID11495791_abstract.ann...
parsing PMID11525413_abstract.ann...
parsing PMID1161793_abstract.ann...
parsing PMID12060229_abstract.ann...
parsing PMID12215499_abstract.ann...
parsing PMID12223598_abstract.ann...
parsing PMID12445118_abstract.ann...
parsing PMID12709495_abstract.ann...
parsing PMID12744517_abstract.ann...
parsing PMID12746525_abstract.ann...
parsing PMID12750377_abstract.ann...
parsing PMID12754834_abstract.ann...
parsing PMID12777041_abstract.ann...
parsing PMID12825696_abstract.ann...
parsing PMID14555690_abstract.ann...
parsing PMID14676400_abstract.ann...
pa

In [13]:
# write entities & relations to json files

with open(outfile_entities, "w", encoding = "utf-8") as out:
    json.dump(entities, out, ensure_ascii = False, indent = 4)

with open(outfile_relations, "w", encoding = "utf-8") as out:
    json.dump(relations, out, ensure_ascii = False, indent = 4)


In [14]:
# shuffle & split triples

rd_state = 7
train, rest = train_test_split(triples, test_size = 0.2, train_size = 0.8, random_state = rd_state, shuffle = True)
val, test = train_test_split(rest, test_size = 0.5, train_size = 0.5, random_state = rd_state, shuffle = False)

# get set sizes
print(f"overall = {len(triples)} triples")
print(f"train = {len(train)} triples (80%)")
print(f"validation = {len(val)} triples (10%)")
print(f"test = {len(test)} triples (10%)")

overall = 2149 triples
train = 1719 triples (80%)
validation = 215 triples (10%)
test = 215 triples (10%)


In [15]:
# write train, test, validation sets to txt files

with open(outfile_train, "w", encoding = "utf-8", newline = "\n") as out:
    out.writelines(train)

with open(outfile_val, "w", encoding = "utf-8", newline = "\n") as out:
    out.writelines(val)

with open(outfile_test, "w", encoding = "utf-8", newline = "\n") as out:
    out.writelines(test)